# 第24章　消化器 ― CT肝胆膵の統合セグメンテーションと腫瘍検出**『本格実装 医療診断支援AI（実装編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-impl

## 実装 ― 知識を構造化し、後処理として課す

```json{  "containment": [    {"child": "pancreatic_tumor", "parents": ["pancreas", "duodenum"], "margin_mm": 12},    {"child": "liver_tumor",      "parents": ["liver"],                "margin_mm": 8},    {"child": "gallstone",        "parents": ["gallbladder"],          "margin_mm": 6}  ],  "keep_largest_component": ["liver", "spleen", "pancreas", "gallbladder"],  "remove_small_components": [    {"label": "pancreatic_tumor", "min_volume_mm3": 50},    {"label": "liver_tumor",      "min_volume_mm3": 50}  ]}```

In [ ]:
LABELS = {"liver": 1, "spleen": 2, "pancreas": 3, "duodenum": 5, "gallbladder": 6,          "pancreatic_tumor": 18, "liver_tumor": 22, "gallstone": 25}   # 24.4節の体系

In [ ]:
import numpy as np, nibabel as nibfrom scipy import ndimagedef spacing_mm(affine):    return tuple(float(np.linalg.norm(affine[:3, i])) for i in range(3))def apply_containment(seg, child, parents, margin_mm, affine):    """子ラベル（腫瘍・結石）が、親臓器から margin_mm 以上離れていたら、その塊ごと消す。"""    child_mask = (seg == child)    if not child_mask.any():        return seg, 0    parent_mask = np.zeros_like(child_mask)    for p in parents:        parent_mask |= (seg == p)    if not parent_mask.any():        # 親臓器が一つも予測されていない＝包含を判定できない。        # ここで腫瘍を消すと、臓器の予測失敗が腫瘍の巻き添えになる。何もしないのが安全側。        return seg, 0    # 距離は「物理的なmm」で測る。CTは 0.7×0.7×5.0mm のように異方性なので、    # ボクセル数で測るとz方向のmarginが7倍甘くなる（よくある事故）。    dist = ndimage.distance_transform_edt(~parent_mask, sampling=spacing_mm(affine))    lbl, n = ndimage.label(child_mask, structure=np.ones((3, 3, 3)))    removed = 0    for c in range(1, n + 1):        comp = (lbl == c)        if dist[comp].min() > margin_mm:      # 塊「全体」が margin の外にあるときだけ消す            seg[comp] = 0                      # 一部でも臓器に接していれば残す（保守的に）            removed += int(comp.sum())    return seg, removed

## 学習の中で制約する ― 包含損失

In [ ]:
import torchdef containment_loss(tumor_prob, organ_region):    """腫瘍の確率が、臓器の外に漏れ出したぶんを罰する。    tumor_prob   : 腫瘍クラスの確率マップ (B, D, H, W)    organ_region : 「腫瘍が居てよい範囲」を表す二値マップ (B, D, H, W)。                   教師の臓器マスク（または別に学習した臓器モデルの出力）を、                   許容マージンぶん膨張させて作る。"""    return (tumor_prob * (1.0 - organ_region)).mean()# 許容領域は「臓器とその臓器内の腫瘍を合わせた範囲」。排他ラベルでは臓器マスクに腫瘍の穴が空くので、# 臓器だけを膨張させると、大きな腫瘍の中心が許容外に落ちて正解ほど罰せられる。organ_region = dilate(organ_gt | tumor_gt, margin_mm=10)   # 和集合をマージンぶん膨張loss = dice_ce_loss(pred, target) + 0.1 * containment_loss(tumor_p, organ_region)

## がん検出の外へ ― 肝の「量」を測る

In [ ]:
def hepatic_steatosis_ct(liver_hu, spleen_hu, rule):    """rule は採用する判定基準（名称・版・対象・除外条件・計測法・出典）を持つ設定。    基準値をコードに直書きすると、どの基準を実装したのか後から追えなくなる。    ここで返すのは計測値と『採用したルールでの区分』であって、診断名ではない。"""    ls_diff = liver_hu - spleen_hu             # 単純CTでの肝-脾差（負ほど脂肪多い）    band = rule.classify(ls_diff)              # 例: {"name": ..., "version": ..., "cutoffs": ...}    return {"liver_hu": liver_hu, "L_minus_S": round(ls_diff, 1),            "rule": rule.name, "rule_version": rule.version, "band": band}

## 転移を数える ― リンパ節という難物

In [ ]:
def node_equiv_ellipse_minor_axis_mm(node_mask, mm_per_px):    """等価楕円の短軸長[mm]：二次モーメントが同じ楕円の短軸（skimage の minor_axis_length と同じ定義）。    輪郭上で実測する臨床の短径とは別物で、形状によりずれる（長方形なら幅の約1.15倍）。    mm_per_px は (行方向, 列方向) の2要素。非等方画素にも対応する。"""    ys, xs = np.where(node_mask)    if len(ys) < 3:        return None                                       # 空・1〜2画素は評価不能    dy, dx = (float(mm_per_px), float(mm_per_px)) if np.isscalar(mm_per_px) else map(float, mm_per_px)    pts = np.stack([xs * dx, ys * dy], 1)                 # 先に物理座標[mm]へ直す    pts -= pts.mean(0)    _, s, _ = np.linalg.svd(pts, full_matrices=False)     # 主軸方向の広がり    # 短軸長 = 4σ_min。2σ だと「半径」相当になり、10mmの節が5mmと出るので注意    return round(4.0 * s[-1] / np.sqrt(len(pts)), 1)

## 切る前に組み立てる ― 術前計画のための3D

In [ ]:
# 判定の基準は関数に埋め込まず、基準名・対象・適用外を持つ「規則」として渡す。# 数値は施設・ガイドラインの基準に置き換える（下の 25 は健常肝でよく使われる目安の例）FLR_RULE_HEALTHY = {"name": "FLR-healthy-liver", "min_flr_pct": 25,                    "applies_to": "背景肝が健常な症例",                    "not_for": "肝硬変・高度脂肪肝・化学療法後（別の基準を使う）"}import numbers   # NumPy のスカラー型（mask.sum() の返り値など）も実数として受け付けるためdef future_liver_remnant(remnant_voxels, tumor_voxels, total_liver_voxels, voxel_ml, rule):    """total_liver_voxels は「腫瘍を含む全肝」の前提。排他ラベルで肝実質だけを数えた値を    渡すと、腫瘍を二重に引くことになる。集合の定義を呼ぶ側と揃えてから使う。"""    NA = {"remnant_ml": None, "flr_pct": None, "high_risk": None, "rule": rule["name"]}    # 計測値と規則の閾値は、まず数値として有効かを検査する。NaN は大小比較がすべて偽になるので、    # 検査せずに進めると「NaN → high_risk=False, state=ok」という有効な臨床区分に化ける    vals = (remnant_voxels, tumor_voxels, total_liver_voxels, voxel_ml)    if not all(isinstance(v, numbers.Real) and not isinstance(v, bool) and np.isfinite(v) for v in vals):        return {**NA, "state": "評価不能（計測値が数値でない、または非有限）"}    if not (0 <= rule["min_flr_pct"] <= 100) or not np.isfinite(rule["min_flr_pct"]):        return {**NA, "state": "評価不能（規則の閾値が不正）"}    if voxel_ml <= 0 or tumor_voxels < 0 or total_liver_voxels <= 0 or tumor_voxels > total_liver_voxels:        return {**NA, "state": "評価不能（ボクセル体積・ボクセル数が不正）"}    functional = total_liver_voxels - tumor_voxels          # 腫瘍を含む全肝から腫瘍を除き、非腫瘍部の容積を求める    # 不正な入力を max(...,1) で隠すと、FLR 1000% が high_risk=False になって返る。    if functional <= 0 or remnant_voxels < 0 or remnant_voxels > functional:        return {**NA, "state": "評価不能（非腫瘍部が0以下、または残肝が非腫瘍部を超える）"}    flr_pct = 100.0 * remnant_voxels / functional    return {"remnant_ml": round(remnant_voxels * voxel_ml, 1),            "flr_pct": round(flr_pct, 1),            "rule": rule["name"], "applies_to": rule["applies_to"],   # 判定は規則名と一緒に返す            "high_risk": bool(flr_pct < rule["min_flr_pct"]),            "state": "ok"}